# User Classification Analytics Lab

Additional user/product analytics lab notebook included as a reviewable portfolio artifact.

This public portfolio copy keeps the full notebook source visible on GitHub while removing execution outputs, execution counts, and environment-specific metadata.

# Lesson 3 — User Analytics from Scratch
### MSc Data Science & Economics · 5 May 2026

---

## The problem

You have been given a raw payments table — exactly as it comes out of a payment processor's event log. There is no user layer. There is no clean `user_id`. There are identity signals: email, phone, device fingerprint, user agent, IP address.

Your job in this notebook is to **build the user layer from scratch**.

By the end you will have written functions that:
1. Normalise and resolve identity signals into a
 stable `user_id`
2. Assign each user a **data tier** (0–3) based on available signals
3. Classify users into **lifecycle stages** (new, returning, churned...)
4. Assign **activity frequency tiers** (Champion → Churned)
5. Compute **RFM scores** (Recency, Frequency, Monetary)
6. Build the **growth accounting** table (New + Resurrected − Churned)

---

### Dataset

**`raw_payments.csv`** — ~3,500 payment events across 5 merchants, Nov 2025–May 2026.

Columns you will work with:

| Column | Description |
|--------|-------------|
| `txn_id` | Unique transaction identifier |
| `timestamp` | Event datetime |
| `merchant_id`, `merchant_name`, `merchant_category` | Merchant info |
| `raw_user_id` | Merchant-generated ID — **not reliable across merchants** |
| `email` | Captured email — may be missing, uppercase, aliased |
| `phone` | Captured phone — often missing |
| `device_fingerprint` | Browser/device hash — changes per device |
| `device_type` | e.g. `mobile_ios`, `desktop_chrome` |
| `user_agent` | Raw browser string |
| `ip_address` | Network IP — can be shared or VPN |
| `payment_method` | `card` or `open_banking` |
| `amount_gbp` | Transaction amount |
| `response_code` | Auth response code |
| `is_settled` | Whether payment completed |
| `settlement_date` | Date funds settled (if settled) |

> **Reference date for all recency calculations: 5 May 2026**

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from collections import defaultdict

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

TODAY = pd.Timestamp('2026-05-05')

raw = pd.read_csv('raw_payments.csv', parse_dates=['timestamp', 'settlement_date'])

print(f'Rows          : {len(raw):,}')
print(f'Date range    : {raw["timestamp"].min().date()} → {raw["timestamp"].max().date()}')
print(f'Merchants     : {raw["merchant_name"].nunique()}')
print(f'raw_user_ids  : {raw["raw_user_id"].nunique():,}')
print()
print(raw.head(3).T)

---
## Section 1 — Understanding the identity signals

Before writing any functions, explore what we have.

### 1.1 — Signal availability per merchant

For each merchant, calculate:
- Total transactions
- % of rows with `email` present
- % of rows with `phone` present
- % of rows with a real `device_fingerprint` (not `00000000ffff` — our incognito placeholder)

**Why this matters:** the signals available to you vary by merchant. FreshBasket barely captures email. StreamPro always does. This determines what deduplication is even possible.

In [ ]:
# YOUR CODE


#### ✅ Answer — 1.1

In [ ]:
signal_quality = raw.groupby('merchant_name').apply(lambda g: pd.Series({
    'txns':            len(g),
    'email_pct':       round(g['email'].notna().mean() * 100, 1),
    'phone_pct':       round(g['phone'].notna().mean() * 100, 1),
    'real_device_pct': round((g['device_fingerprint'] != '00000000ffff').mean() * 100, 1),
})).reset_index()

print(signal_quality.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(signal_quality))
w = 0.25
ax.bar(x - w,   signal_quality['email_pct'],       w, label='Email %',          color='#163D6E')
ax.bar(x,       signal_quality['phone_pct'],       w, label='Phone %',          color='#0D9488')
ax.bar(x + w,   signal_quality['real_device_pct'], w, label='Real device fp %', color='#F59E0B')
ax.set_xticks(x)
ax.set_xticklabels(signal_quality['merchant_name'], rotation=10)
ax.set_ylabel('% of transactions')
ax.set_title('Identity Signal Availability by Merchant')
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.show()

print()
print('Key observation: FreshBasket captures email on only ~32% of transactions.')
print('For ~68% of FreshBasket transactions, our only signal is device fingerprint.')
print('This means cross-merchant deduplication is nearly impossible for those users.')

### 1.2 — The email mess

Look at the email column. You will find:
- Uppercase versions: `JOHN.SMITH@GMAIL.COM`
- Gmail dot variants: `john.smith@gmail.com` vs `johnsmith@gmail.com`
- Gmail alias: `@gmail.com` vs `@googlemail.com`
- Mixed case: `John.Smith@gmail.com`

How many unique email values exist in the raw data? How many do you expect after normalisation?

Write a function `normalise_email(email)` that:
1. Lowercases
2. Strips whitespace
3. Replaces `@googlemail.com` → `@gmail.com`
4. For Gmail addresses only: removes dots from the local part

Apply it. How many unique emails remain?

In [ ]:
# YOUR CODE


#### ✅ Answer — 1.2

In [ ]:
print(f'Unique raw emails   : {raw["email"].dropna().nunique():,}')

# Show a few examples of the problem
sample = raw[raw['email'].notna()].copy()
dup_examples = sample[sample['email'].str.contains('gmail|googlemail', case=False, na=False)]['email'].head(15)
print('\nSample emails (raw):')
print(dup_examples.values)

def normalise_email(email: str) -> str:
    """Normalise an email address to a canonical form."""
    if pd.isna(email):
        return None
    # Step 1: lowercase and strip
    email = email.lower().strip()
    # Step 2: googlemail → gmail
    email = email.replace('@googlemail.com', '@gmail.com')
    # Step 3: remove dots from Gmail local part only
    if email.endswith('@gmail.com'):
        local, domain = email.split('@')
        local = local.replace('.', '')
        email = f'{local}@{domain}'
    return email

raw['email_normalised'] = raw['email'].apply(normalise_email)

print(f'\nUnique emails BEFORE normalisation: {raw["email"].dropna().nunique():,}')
print(f'Unique emails AFTER  normalisation: {raw["email_normalised"].dropna().nunique():,}')
print(f'Duplicates eliminated             : {raw["email"].dropna().nunique() - raw["email_normalised"].dropna().nunique():,}')
print()
print('These eliminated duplicates = same real person being counted as 2 different users.')

---
## Section 2 — Building a stable user_id

The `raw_user_id` in the dataset is generated per-merchant from whatever identity signal was available at payment time. The same person can have different `raw_user_id` values across merchants — or even within the same merchant if they used a different email alias.

### 2.1 — Understand the raw_user_id problem

Find real evidence of the over-counting problem. Look for cases where:
- The same normalised email appears under **multiple different `raw_user_id`s**

How many people does this affect? How many phantom duplicate user IDs does it create?

In [ ]:
# YOUR CODE


#### ✅ Answer — 2.1

In [ ]:
# For each normalised email, count distinct raw_user_ids
email_to_rawids = (
    raw[raw['email_normalised'].notna()]
    .groupby('email_normalised')['raw_user_id']
    .nunique()
    .reset_index()
    .rename(columns={'raw_user_id': 'n_raw_ids'})
)

fragmented = email_to_rawids[email_to_rawids['n_raw_ids'] > 1]

print(f'Emails with 1 raw_user_id   : {(email_to_rawids["n_raw_ids"]==1).sum():,}  (clean)')
print(f'Emails with 2+ raw_user_ids : {len(fragmented):,}  ← same person, multiple IDs')
print(f'Total phantom duplicate IDs : {fragmented["n_raw_ids"].sum() - len(fragmented):,}')
print()
print('Examples:')
print(fragmented.head(5).to_string(index=False))

# Show a concrete example
example_email = fragmented.iloc[0]['email_normalised']
example_rows  = raw[raw['email_normalised'] == example_email][['txn_id','merchant_name','raw_user_id','email','device_fingerprint']]
print(f'\nConcrete example for {example_email}:')
print(example_rows.drop_duplicates('raw_user_id').to_string(index=False))
print()
print('These are the same person. Without normalisation, they count as 2 new users.')

### 2.2 — Write the `build_user_id` function

Now build a function that creates a **stable, canonical `user_id`** for each transaction row.

Resolution priority (use the first signal that is available):

1. **Normalised email** — highest reliability, cross-device, cross-merchant
2. **Phone number** — high reliability, cross-device
3. **Device fingerprint** — medium reliability (single device only), only if not incognito
4. **`raw_user_id`** — fallback, merchant-scoped only

The function should return a short hash (first 16 chars of MD5) so it looks like a real system ID.

Apply it to the full dataset and add a `user_id` column.

In [ ]:
# YOUR CODE


#### ✅ Answer — 2.2

In [ ]:
INCOGNITO_FP = '00000000ffff'

def build_user_id(row) -> str:
    """
    Derive a stable user_id from available identity signals.
    Priority: normalised email > phone > device fingerprint > raw_user_id
    """
    # 1. Normalised email
    if pd.notna(row.get('email_normalised')):
        signal = f"email::{row['email_normalised']}"
    # 2. Phone
    elif pd.notna(row.get('phone')):
        signal = f"phone::{row['phone']}"
    # 3. Device fingerprint (not incognito)
    elif pd.notna(row.get('device_fingerprint')) and row['device_fingerprint'] != INCOGNITO_FP:
        signal = f"device::{row['device_fingerprint']}"
    # 4. Fallback to raw_user_id
    else:
        signal = f"raw::{row['raw_user_id']}"

    return hashlib.md5(signal.encode()).hexdigest()[:16]


raw['user_id'] = raw.apply(build_user_id, axis=1)

print(f'raw_user_id count (before) : {raw["raw_user_id"].nunique():,}')
print(f'user_id count (after)      : {raw["user_id"].nunique():,}')
print(f'Reduction                  : {raw["raw_user_id"].nunique() - raw["user_id"].nunique():,} phantom duplicates collapsed')
print()

# Which signal did we actually use per row?
def signal_used(row):
    if pd.notna(row.get('email_normalised')):
        return 'email'
    elif pd.notna(row.get('phone')):
        return 'phone'
    elif pd.notna(row.get('device_fingerprint')) and row['device_fingerprint'] != INCOGNITO_FP:
        return 'device_fp'
    return 'raw_user_id'

raw['id_signal_used'] = raw.apply(signal_used, axis=1)
print('Signal used per transaction:')
print(raw['id_signal_used'].value_counts())
print()
print('Transactions where only raw_user_id was available are the least trustworthy.')
print('These user_ids are merchant-scoped — they cannot be linked across merchants.')

### 2.3 — Build the user table

Now that every transaction has a `user_id`, collapse to a **one-row-per-user** table. For each user, capture the best available version of each signal.

Your user table should have:
- `user_id`
- `email_normalised` (first non-null)
- `phone` (first non-null)
- `device_fingerprint` (most common non-incognito)
- `id_signal_used` (best signal quality)
- `first_seen` (earliest timestamp)
- `last_seen` (latest timestamp)
- `txn_count` (total attempts)
- `merchant_count` (distinct merchants)

> This is the foundation table. Every other analysis in the lesson builds on it.

In [ ]:
# YOUR CODE


#### ✅ Answer — 2.3

In [ ]:
def best_value(series):
    """Return first non-null value, or None."""
    vals = series.dropna()
    return vals.iloc[0] if len(vals) > 0 else None

def best_device_fp(series):
    """Most common non-incognito fingerprint."""
    real = series[series != INCOGNITO_FP].dropna()
    return real.mode().iloc[0] if len(real) > 0 else None

def best_signal(series):
    """Return the highest-reliability signal used for this user."""
    priority = {'email': 0, 'phone': 1, 'device_fp': 2, 'raw_user_id': 3}
    return min(series.dropna(), key=lambda x: priority.get(x, 99), default='raw_user_id')

users = raw.groupby('user_id').agg(
    email           = ('email_normalised',  best_value),
    phone           = ('phone',             best_value),
    device_fp       = ('device_fingerprint',best_device_fp),
    best_id_signal  = ('id_signal_used',    best_signal),
    first_seen      = ('timestamp',         'min'),
    last_seen       = ('timestamp',         'max'),
    txn_count       = ('txn_id',            'count'),
    merchant_count  = ('merchant_id',       'nunique'),
    device_types    = ('device_type',       lambda x: ','.join(sorted(x.unique()))),
).reset_index()

print(f'User table: {len(users):,} rows')
print()
print(users.head(3).T)
print()
print('Best identity signal distribution:')
print(users['best_id_signal'].value_counts())

---
## Section 3 — Data Tier Assignment

### 3.1 — Write the `assign_data_tier` function

From section 3.2 of the slides:

| Tier | Condition |
|------|-----------|
| 0 | No reliable identity signal — only `raw_user_id` fallback |
| 1 | Has device_fp or phone (can attempt payment, basic dedup possible) |
| 2 | Has email or phone (enables cross-merchant linking, re-engagement) |
| 3 | Has email AND phone (highest confidence — full re-engagement possible) |

Write the function and apply it to the user table. Show the distribution.

In [ ]:
# YOUR CODE


#### ✅ Answer — 3.1

In [ ]:
def assign_data_tier(row) -> int:
    """
    Assign data tier 0–3 based on available identity signals.
    Tier 3 = email + phone (highest confidence)
    Tier 2 = email or phone
    Tier 1 = device fingerprint only (can transact, can't re-engage)
    Tier 0 = only raw fallback — lowest confidence
    """
    has_email  = pd.notna(row['email'])
    has_phone  = pd.notna(row['phone'])
    has_device = pd.notna(row['device_fp'])

    if has_email and has_phone:
        return 3
    if has_email or has_phone:
        return 2
    if has_device:
        return 1
    return 0

users['data_tier'] = users.apply(assign_data_tier, axis=1)

tier_labels = {0: 'Tier 0\n(Raw only)', 1: 'Tier 1\n(Device)', 2: 'Tier 2\n(Email/Phone)', 3: 'Tier 3\n(Full)'}
tier_dist   = users['data_tier'].value_counts().sort_index()
tier_dist_pct = (tier_dist / len(users) * 100).round(1)

print('Data tier distribution:')
for t, c in tier_dist.items():
    print(f'  Tier {t}: {c:>5} users ({tier_dist_pct[t]}%)')

colors = ['#DC2626','#163D6E','#F59E0B','#16A34A']
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([tier_labels[t] for t in tier_dist.index],
               tier_dist_pct.values, color=colors)
ax.set_ylabel('% of users')
ax.set_title('User Distribution by Data Tier')
for bar, v in zip(bars, tier_dist_pct.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print()
print('Analyst note: Tier 0 users are effectively invisible for analytics.')
print('Only Tier 2+ users can be re-engaged via email or SMS campaigns.')

---
## Section 4 — Lifecycle & Activity Tier Classification

Now join the settled transaction history back to users and classify each user's engagement.

### 4.1 — Compute user activity metrics

First, build a settled payments summary per user (observation window = last 12 months from today):
- `settled_count` — number of settled payments
- `total_spend` — total settled amount
- `avg_basket` — average basket value
- `first_payment` — earliest settled payment date
- `last_payment` — most recent settled payment date
- `days_since_last` — days between last payment and today

In [ ]:
# YOUR CODE


#### ✅ Answer — 4.1

In [ ]:
OBS_START = TODAY - pd.Timedelta(days=365)

settled = raw[
    (raw['is_settled'] == True) &
    (raw['settlement_date'] >= OBS_START) &
    (raw['settlement_date'] <= TODAY)
].copy()

activity = settled.groupby('user_id').agg(
    settled_count  = ('txn_id',           'count'),
    total_spend    = ('amount_gbp',        'sum'),
    avg_basket     = ('amount_gbp',        'mean'),
    first_payment  = ('settlement_date',   'min'),
    last_payment   = ('settlement_date',   'max'),
).reset_index()

activity['days_since_last'] = (TODAY - activity['last_payment']).dt.days
activity['total_spend']     = activity['total_spend'].round(2)
activity['avg_basket']      = activity['avg_basket'].round(2)

# Merge onto user table
users = users.merge(activity, on='user_id', how='left')
users['settled_count']   = users['settled_count'].fillna(0).astype(int)
users['total_spend']     = users['total_spend'].fillna(0)
users['days_since_last'] = users['days_since_last'].fillna(9999).astype(int)

print(f'Users with ≥1 settled payment  : {(users["settled_count"]>0).sum():,}')
print(f'Users with 0 settled payments  : {(users["settled_count"]==0).sum():,}')
print()
print(activity.describe().round(1))

### 4.2 — Write the `assign_lifecycle_stage` function

Implement the lifecycle stages from section 3.2:

| Stage | Rule |
|-------|------|
| `Never Paid` | Has user_id but 0 settled payments ever |
| `New User` | First settled payment within last 30 days |
| `New Merchant User` | First payment at this merchant within last 30 days (ignore for now — user-level only) |
| `Returning User` | ≥2 payments, last payment within 90 days |
| `Inactive` | Has payments, last payment 91–365 days ago |
| `Churned` | Last payment > 365 days ago |


In [ ]:
# YOUR CODE


#### ✅ Answer — 4.2

In [ ]:
def assign_lifecycle_stage(row) -> str:
    if row['settled_count'] == 0:
        return 'Never Paid'

    days_since = row['days_since_last']
    first_payment = row.get('first_payment')
    days_since_first = (TODAY - pd.Timestamp(first_payment)).days if pd.notna(first_payment) else 9999

    if days_since > 365:
        return 'Churned'
    if days_since > 90:
        return 'Inactive'
    if days_since_first <= 30 and row['settled_count'] <= 2:
        return 'New User'
    return 'Returning User'

users['lifecycle_stage'] = users.apply(assign_lifecycle_stage, axis=1)

lc_counts = users['lifecycle_stage'].value_counts()
lc_pct    = (lc_counts / len(users) * 100).round(1)

STAGE_ORDER  = ['New User','Returning User','Inactive','Churned','Never Paid']
STAGE_COLORS = ['#16A34A','#163D6E','#F59E0B','#DC2626','#94A3B8']

fig, ax = plt.subplots(figsize=(10, 4))
counts_ordered = [lc_counts.get(s,0) for s in STAGE_ORDER]
pcts_ordered   = [lc_pct.get(s,0) for s in STAGE_ORDER]
bars = ax.barh(STAGE_ORDER[::-1], counts_ordered[::-1], color=STAGE_COLORS[::-1])
ax.set_xlabel('Number of users')
ax.set_title('User Lifecycle Stage Distribution (as of 5 May 2026)')
for i, (c, p) in enumerate(zip(counts_ordered[::-1], pcts_ordered[::-1])):
    ax.text(c+5, i, f'{c:,} ({p}%)', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### 4.3 — Write the `assign_activity_tier` function

From section 3.3 of the slides — based on **settled payment count in the last 12 months**:

| Tier | Threshold | Frequency |
|------|-----------|----------|
| Champion | ≥ 52 payments | ≥ 1/week |
| Super | ≥ 12 | ≥ 1/month |
| Engaged | ≥ 4 | ≥ 1/quarter |
| Light | ≥ 1 | At least once in 6 months |
| Inactive | 0 in 12m, but paid in 6–12m window | — |
| Churned | No payment for > 12 months | — |

Remember: the tier is **dynamic** — computed from today's view, not from when the user first appeared.

In [ ]:
# YOUR CODE


#### ✅ Answer — 4.3

In [ ]:
def assign_activity_tier(row) -> str:
    """
    Activity tier based on settled payment count in last 12 months
    and recency of last settled payment.
    The tier is relative to TODAY — not to the user's signup date.
    """
    n     = row['settled_count']     # payments in last 12 months
    days  = row['days_since_last']   # days since last payment

    if days > 365:
        return 'Churned'
    if n == 0 or days > 180:
        return 'Inactive'
    if n >= 52:
        return 'Champion'
    if n >= 12:
        return 'Super'
    if n >= 4:
        return 'Engaged'
    return 'Light'

users['activity_tier'] = users.apply(assign_activity_tier, axis=1)

TIER_ORDER  = ['Champion','Super','Engaged','Light','Inactive','Churned']
TIER_COLORS = ['#7C3AED','#163D6E','#0D9488','#F59E0B','#F97316','#DC2626']

tier_counts = users['activity_tier'].value_counts().reindex(TIER_ORDER, fill_value=0)
tier_pct    = (tier_counts / len(users) * 100).round(1)

# Join with TPV to build Pareto chart
user_tpv = settled.groupby('user_id')['amount_gbp'].sum().reset_index()
users_with_tpv = users.merge(user_tpv, on='user_id', how='left')
users_with_tpv['amount_gbp'] = users_with_tpv['amount_gbp'].fillna(0)

tier_tpv = users_with_tpv.groupby('activity_tier')['amount_gbp'].sum().reindex(TIER_ORDER, fill_value=0)
tier_tpv_pct = (tier_tpv / tier_tpv.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x, w = np.arange(len(TIER_ORDER)), 0.38
axes[0].bar(x - w/2, tier_pct.values,     w, label='% of users', color=TIER_COLORS, alpha=0.65)
axes[0].bar(x + w/2, tier_tpv_pct.values, w, label='% of TPV',   color=TIER_COLORS, alpha=1.0)
axes[0].set_xticks(x)
axes[0].set_xticklabels(TIER_ORDER, rotation=10)
axes[0].set_ylabel('% share')
axes[0].set_title('Activity Tiers: % Users vs % TPV (Pareto)')
axes[0].legend()

# Summary table
summary = pd.DataFrame({'users': tier_counts, 'user_pct': tier_pct, 'tpv_pct': tier_tpv_pct})
print(summary.to_string())

# Pie of activity tiers
axes[1].pie(tier_counts.values, labels=TIER_ORDER, colors=TIER_COLORS,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Activity Tier Distribution')

plt.tight_layout()
plt.show()

top2_u = tier_pct[['Champion','Super']].sum()
top2_t = tier_tpv_pct[['Champion','Super']].sum()
print(f'\nChampion + Super: {top2_u:.1f}% of users → {top2_t:.1f}% of TPV')

---
## Section 5 — RFM Scoring

### 5.1 — Compute R, F, M raw values

For each user with at least one settled payment in the observation window:

| Dimension | Field | Description |
|-----------|-------|-------------|
| **R** (Recency) | `days_since_last` | Days since last settled payment |
| **F** (Frequency) | `settled_count` | Count of settled payments in 12m |
| **M** (Monetary) | `total_spend` | Sum of settled payment amounts in 12m |

Then score each dimension 1–5 using **quintile buckets** (`pd.qcut`). Score 5 = best for the merchant.
- R: fewer days = score 5 (most recent)
- F: more payments = score 5
- M: more spend = score 5

Combine into `RFM_code` (e.g. `"543"`) and `RFM_total` (sum of scores).

In [ ]:
# YOUR CODE


#### ✅ Answer — 5.1

In [ ]:
def score_rfm(users_df: pd.DataFrame) -> pd.DataFrame:
    """
    Score R, F, M dimensions 1–5 using quintile buckets.
    Only users with ≥1 settled payment are scored.
    """
    scored = users_df[users_df['settled_count'] > 0].copy()

    # R: reversed — fewer days since last = score 5
    scored['R'] = pd.qcut(
        scored['days_since_last'].clip(upper=365),
        q=5, labels=[5, 4, 3, 2, 1], duplicates='drop'
    ).astype(int)

    # F: more payments = score 5
    scored['F'] = pd.qcut(
        scored['settled_count'].rank(method='first'),
        q=5, labels=[1, 2, 3, 4, 5]
    ).astype(int)

    # M: more spend = score 5
    scored['M'] = pd.qcut(
        scored['total_spend'].rank(method='first'),
        q=5, labels=[1, 2, 3, 4, 5]
    ).astype(int)

    scored['RFM_code']  = scored['R'].astype(str) + scored['F'].astype(str) + scored['M'].astype(str)
    scored['RFM_total'] = scored['R'] + scored['F'] + scored['M']

    return scored

rfm = score_rfm(users)
users = users.merge(rfm[['user_id','R','F','M','RFM_code','RFM_total']], on='user_id', how='left')

print(f'Users with RFM scores: {rfm["RFM_code"].notna().sum():,}')
print()
print('Score distribution:')
print(rfm[['R','F','M']].describe().round(2))
print()
print('Top 10 RFM codes:')
print(rfm['RFM_code'].value_counts().head(10))

### 5.2 — Write the `assign_rfm_segment` function

Map RFM scores to named segments (from section 3.4 of the slides):

| Segment | Rule |
|---------|------|
| Champions | R≥4, F≥4, M≥4 |
| Loyal Users | R≥4, F≥4 |
| Promising | R=5, F≤2 |
| At Risk | R≤2, F≥4, M≥4 |
| Needs Attention | R=3, F≤2 |
| Hibernating | R≤2, F≤2, M≥3 |
| Lost | R≤2, F≤2, M≤2 |
| New Users | R=5, F=1, M=1 |

Then: show the business action for each segment (from the slides).

In [ ]:
# YOUR CODE


#### ✅ Answer — 5.2

In [ ]:
def assign_rfm_segment(row) -> str:
    r, f, m = row.get('R'), row.get('F'), row.get('M')
    if pd.isna(r): return None
    r, f, m = int(r), int(f), int(m)

    if r >= 4 and f >= 4 and m >= 4: return 'Champions'
    if r >= 4 and f >= 4:            return 'Loyal Users'
    if r == 5 and f == 1 and m == 1: return 'New Users'
    if r == 5 and f <= 2:            return 'Promising'
    if r <= 2 and f >= 4 and m >= 4: return 'At Risk'
    if r <= 2 and f <= 2 and m >= 3: return 'Hibernating'
    if r <= 2 and f <= 2 and m <= 2: return 'Lost'
    return 'Needs Attention'

users['RFM_segment'] = users.apply(assign_rfm_segment, axis=1)

ACTIONS = {
    'Champions':      'Reward & protect. Zero friction. High LTV retention focus.',
    'Loyal Users':    'Nurture. Keep friction low. Early feature access.',
    'Promising':      'Activate second payment ASAP. Critical 7-day window.',
    'At Risk':        'Win-back NOW. High-value going silent. Trigger flow before Inactive.',
    'Needs Attention':'Re-engagement prompts. Test frequency incentives.',
    'Hibernating':    'Low-cost channel only. Low recovery rate.',
    'Lost':           'Exclude from active base. Analyse exit reasons.',
    'New Users':      'Onboarding quality. First 7 days are critical for retention.',
}
SEG_COLORS = {
    'Champions':'#7C3AED','Loyal Users':'#163D6E','Promising':'#16A34A',
    'At Risk':'#DC2626','Needs Attention':'#F59E0B',
    'Hibernating':'#94A3B8','Lost':'#CBD5E1','New Users':'#0D9488',
}

seg_counts = users['RFM_segment'].value_counts().dropna()

fig, ax = plt.subplots(figsize=(10, 5))
colors = [SEG_COLORS.get(s,'#888') for s in seg_counts.index]
ax.barh(seg_counts.index, seg_counts.values, color=colors)
ax.set_title('RFM Segment Distribution')
ax.set_xlabel('Number of users')
for i, v in enumerate(seg_counts.values):
    ax.text(v+1, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nRFM Segments + Business Actions:')
print('='*70)
for seg, count in seg_counts.items():
    pct = count / seg_counts.sum() * 100
    print(f'\n{seg:20} ({count} users, {pct:.1f}%)')
    print(f'  → {ACTIONS.get(seg,"")}')

---
## Section 6 — Net New Active Users

### 6.1 — Build the growth accounting function

Write a function `compute_growth_accounting(payments_df, active_window_days=90)` that, for each month in the dataset, returns:

| Column | Definition |
|--------|------------|
| `new` | Users whose **first-ever** settled payment was in this month |
| `resurrected` | Users who paid this month, previously inactive for ≥ `active_window_days` |
| `churned` | Users who were active last month but have not paid in ≥ `active_window_days` |
| `nna` | `new + resurrected − churned` |
| `active` | Unique users with a settled payment in this month |

Then plot the waterfall chart.

In [ ]:
# YOUR CODE


#### ✅ Answer — 6.1

In [ ]:
def compute_growth_accounting(payments_df: pd.DataFrame,
                               active_window_days: int = 90) -> pd.DataFrame:
    """
    Compute monthly growth accounting from settled payments.

    New:         first-ever settled payment this month
    Resurrected: paid this month, previously silent ≥ active_window_days
    Churned:     active last month, now silent ≥ active_window_days
    NNA:         New + Resurrected − Churned
    """
    df = payments_df[payments_df['is_settled'] == True].copy()
    df['month'] = df['settlement_date'].dt.to_period('M')

    all_months = sorted(df['month'].unique())
    rows       = []

    for i, month in enumerate(all_months):
        month_start = month.to_timestamp()
        month_end   = (month + 1).to_timestamp() - pd.Timedelta(days=1)
        window_start = month_start - pd.Timedelta(days=active_window_days)

        # Users active this month
        active_this = set(df[df['month'] == month]['user_id'])

        # Users ever active before this month
        ever_before = set(df[df['month'] < month]['user_id'])

        # Users active in the window before this month (recently active)
        recently_active = set(df[
            (df['settlement_date'] >= window_start) &
            (df['settlement_date'] < month_start)
        ]['user_id'])

        # Users active last month
        if i > 0:
            prev_month = all_months[i - 1]
            active_last = set(df[df['month'] == prev_month]['user_id'])
        else:
            active_last = set()

        new         = len(active_this - ever_before)
        resurrected = len(active_this & (ever_before - recently_active))
        churned     = len(active_last - recently_active - active_this) if i > 0 else 0
        nna         = new + resurrected - churned

        rows.append({
            'month':        str(month),
            'active':       len(active_this),
            'new':          new,
            'resurrected':  resurrected,
            'churned':      churned,
            'nna':          nna,
        })

    return pd.DataFrame(rows)


growth = compute_growth_accounting(raw)
print(growth.to_string(index=False))

# Waterfall chart
g = growth.copy()
fig, ax1 = plt.subplots(figsize=(12, 5))
x  = np.arange(len(g))
w  = 0.5

ax1.bar(x, g['new'],         w, label='New',         color='#16A34A', alpha=0.9)
ax1.bar(x, g['resurrected'], w, bottom=g['new'],      label='Resurrected', color='#163D6E', alpha=0.9)
ax1.bar(x, -g['churned'],    w,                       label='Churned',     color='#DC2626', alpha=0.8)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(g['month'], rotation=30, fontsize=9)
ax1.set_ylabel('Users')
ax1.set_title('Growth Accounting: New + Resurrected − Churned')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(x, g['nna'], 'o-', color='#7C3AED', linewidth=2.5, label='NNA')
ax2.set_ylabel('Net New Active', color='#7C3AED')
ax2.tick_params(axis='y', labelcolor='#7C3AED')
ax2.spines['top'].set_visible(False)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

# Health check
latest = g[g['month'] < '2026-05'].iloc[-1]
ratio  = latest['new'] / max(latest['churned'], 1)
health = '✅ Healthy' if ratio >= 1.5 else '⚠️ Monitor'
print(f"\nLatest full month ({latest['month']})")
print(f'  New={latest["new"]}  Resurrected={latest["resurrected"]}  Churned={latest["churned"]}')
print(f'  New:Churned ratio = {ratio:.2f}×  → {health}')
print()
print('Benchmark: healthy product sees New > Churned by 1.5× or more.')

---
## Section 7 — Final User Table

### 7.1 — Assemble and export the complete user profile

You now have all the pieces. Assemble the final user table with every derived field, and export it as `user_profiles_derived.csv`.

Then print a summary in the format a Growth Analyst would present to a product team: one row per activity tier with user count, % of base, avg RFM total, avg spend, % contactable (data tier ≥ 2), and the recommended action.

In [ ]:
# YOUR CODE


#### ✅ Answer — 7.1

In [ ]:
TIER_ACTIONS = {
    'Champion':  'Reward & protect. Invest in retention quality.',
    'Super':     'Nurture. Monitor for slippage. Low friction.',
    'Engaged':   'Frequency nudges. Use-case specific messaging.',
    'Light':     'Second payment is the goal. Habit formation.',
    'Inactive':  'Win-back before Churned. Low-cost re-engagement.',
    'Churned':   'Root cause analysis. Low-cost channels only.',
}

# Save
users.to_csv('user_profiles_derived.csv', index=False)
print(f'Saved user_profiles_derived.csv — {len(users):,} users, {len(users.columns)} columns')
print()

# Summary report
summary = users[users['activity_tier'].isin(TIER_ACTIONS.keys())].groupby('activity_tier').agg(
    users           = ('user_id',       'count'),
    avg_rfm_total   = ('RFM_total',     'mean'),
    avg_spend_12m   = ('total_spend',   'mean'),
    pct_contactable = ('data_tier',     lambda x: (x >= 2).mean() * 100),
).reindex(TIER_ORDER).dropna(how='all').round(1)

summary['user_pct'] = (summary['users'] / summary['users'].sum() * 100).round(1)

print('USER HEALTH SUMMARY — 5 May 2026')
print('='*75)
for tier in TIER_ORDER:
    if tier not in summary.index: continue
    r = summary.loc[tier]
    rfm_str = f"{r['avg_rfm_total']:.1f}" if not pd.isna(r['avg_rfm_total']) else 'n/a'
    print(f'\n  {tier:10} | {int(r["users"]):>5} users ({r["user_pct"]}%) | '
          f'Avg RFM: {rfm_str:>4} | Avg spend: £{r["avg_spend_12m"]:>7,.0f} | '
          f'Contactable: {r["pct_contactable"]:.0f}%')
    print(f'               → {TIER_ACTIONS[tier]}')